# Рекомендации на основе содержания

Модель предсказывает рейтинг. Все статистики рейтингов вычисляются только на обучающей части данных.

## Загрузка данных и разделение на train/test

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

df_movies = pd.read_csv('ml-latest/movies.csv')
df_tags = pd.read_csv('ml-latest/tags.csv')
df_ratings = pd.read_csv('ml-latest/ratings.csv')

# Сначала отделяем тестовые рейтинги. Их rating далее не используется для построения признаков.
train_ratings, test_ratings = train_test_split(
    df_ratings, test_size=0.2, random_state=42
)
train_ratings = train_ratings.reset_index(drop=True)
test_ratings = test_ratings.reset_index(drop=True)

print(f'Рейтингов всего: {len(df_ratings)}')
print(f'train: {len(train_ratings)}, test: {len(test_ratings)}')

Рейтингов всего: 100836
train: 80668, test: 20168


## Текстовые признаки фильмов

In [2]:
# Теги агрегируются до одного текста на фильм. Это предотвращает размножение строк рейтингов.
tags_by_movie = (
    df_tags.assign(tag=df_tags['tag'].fillna('').astype(str))
    .groupby('movieId', as_index=False)['tag']
    .agg(' '.join)
)

movie_content = (
    df_movies[['movieId', 'genres']]
    .merge(tags_by_movie, on='movieId', how='left', validate='one_to_one')
    .assign(
        combined_features=lambda data: (
            data['genres'].fillna('').str.replace('|', ' ', regex=False)
            + ' ' + data['tag'].fillna('')
        )
    )
)

def add_content(ratings):
    data = ratings.merge(
        movie_content[['movieId', 'combined_features']],
        on='movieId',
        how='left',
        validate='many_to_one',
    )
    assert len(data) == len(ratings), 'После объединения число рейтингов изменилось'
    return data

train_data = add_content(train_ratings)
test_data = add_content(test_ratings)
print(f'Строк после объединения: train={len(train_data)}, test={len(test_data)}')

Строк после объединения: train=80668, test=20168


## Статистики пользователей и фильмов

In [3]:
STAT_COLUMNS = [
    'user_rating_mean', 'user_rating_median', 'user_rating_variance', 'user_rating_count',
    'movie_rating_mean', 'movie_rating_median', 'movie_rating_variance', 'movie_rating_count',
]

def rating_statistics(reference_ratings, target_ratings):
    """Строит признаки target только по рейтингам reference."""
    user_stats = (
        reference_ratings.groupby('userId')['rating'].agg(['mean', 'median', 'var', 'size'])
        .rename(columns={
            'mean': 'user_rating_mean',
            'median': 'user_rating_median',
            'var': 'user_rating_variance',
            'size': 'user_rating_count',
        })
        .reset_index()
    )
    movie_stats = (
        reference_ratings.groupby('movieId')['rating'].agg(['mean', 'median', 'var', 'size'])
        .rename(columns={
            'mean': 'movie_rating_mean',
            'median': 'movie_rating_median',
            'var': 'movie_rating_variance',
            'size': 'movie_rating_count',
        })
        .reset_index()
    )

    features = (
        target_ratings[['userId', 'movieId']].reset_index(drop=True)
        .merge(user_stats, on='userId', how='left', validate='many_to_one')
        .merge(movie_stats, on='movieId', how='left', validate='many_to_one')
    )

    # Для новых пользователей и фильмов в test используем глобальные значения из reference.
    global_variance = reference_ratings['rating'].var()
    fill_values = {
        'user_rating_mean': reference_ratings['rating'].mean(),
        'user_rating_median': reference_ratings['rating'].median(),
        'user_rating_variance': global_variance,
        'user_rating_count': 0,
        'movie_rating_mean': reference_ratings['rating'].mean(),
        'movie_rating_median': reference_ratings['rating'].median(),
        'movie_rating_variance': global_variance,
        'movie_rating_count': 0,
    }
    return features[STAT_COLUMNS].fillna(fill_values).astype(float)

def make_oof_statistics(ratings, n_splits=5):
    """OOF-признаки train: рейтинг строки не попадает в её собственные статистики."""
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_features = np.empty((len(ratings), len(STAT_COLUMNS)), dtype=float)

    for reference_idx, validation_idx in splitter.split(ratings):
        reference = ratings.iloc[reference_idx]
        validation = ratings.iloc[validation_idx]
        oof_features[validation_idx] = rating_statistics(reference, validation).to_numpy()

    return pd.DataFrame(oof_features, columns=STAT_COLUMNS)

# Для train — OOF-статистики, для test — статистики по полному train.
train_statistics = make_oof_statistics(train_ratings)
test_statistics = rating_statistics(train_ratings, test_ratings)

print(train_statistics.head())

   user_rating_mean  ...  movie_rating_count
0          3.215548  ...                 9.0
1          4.005376  ...                 2.0
2          3.411565  ...                68.0
3          3.694279  ...                41.0
4          3.417785  ...                21.0

[5 rows x 8 columns]


## Обучение модели и RMSE

In [4]:
# TF-IDF обучается только на фильмах из train; test лишь преобразуется готовым векторизатором.
train_documents = train_data.drop_duplicates('movieId')['combined_features']
vectorizer = TfidfVectorizer()
vectorizer.fit(train_documents)

tfidf_train = vectorizer.transform(train_data['combined_features'])
tfidf_test = vectorizer.transform(test_data['combined_features'])

# Масштабирование числовых признаков также обучается исключительно на train.
scaler = StandardScaler()
statistics_train_scaled = scaler.fit_transform(train_statistics)
statistics_test_scaled = scaler.transform(test_statistics)

X_train = hstack([tfidf_train, csr_matrix(statistics_train_scaled)], format='csr')
X_test = hstack([tfidf_test, csr_matrix(statistics_test_scaled)], format='csr')
y_train = train_ratings['rating']
y_test = test_ratings['rating']

# Ridge устойчивее обычной линейной регрессии при большом числе TF-IDF-признаков.
model = Ridge(alpha=10.0, solver='lsqr')
model.fit(X_train, y_train)

y_pred = np.clip(model.predict(X_test), y_train.min(), y_train.max())
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'Размерность признаков: {X_train.shape[1]}')
print(f'RMSE: {rmse:.4f}')

Размерность признаков: 1740
RMSE: 0.8804
